# CSCI316: Big Data Mining - Modeling and Evaluation (Spark ML)
## Dubai Land Department Transaction Price Prediction

This notebook implements the **modeling + evaluation at scale using Spark / Spark ML**, while preserving the same core ideas from the original work:
- Manual 10-fold cross-validation (no library CV helpers)
- Baseline single model (Decision Tree)
- Bagging ensemble (from scratch)
- Gradient Boosting ensemble (from scratch)
- RMSE and MAE evaluation

**Input**: uses the prepared dataset `Transactions_copy.csv` (already cleaned/preprocessed).

## 1. Load Required Libraries

In [1]:
# --- Colab setup: run this cell first when using Google Colab ---
try:
  import google.colab
  _in_colab = True
except ImportError:
  _in_colab = False

if _in_colab:
  !pip install -q pyspark
  base_dir = "/content"  # Upload Transactions_copy.csv
  print("Colab detected: PySpark installed, base_dir = /content")
  print("Upload 'Transactions_copy.csv' to the file browser (left panel), or set base_dir to your Drive path.")
else:
  import os
  base_dir = os.getcwd()
  print("Local run: base_dir = current working directory")

Local run: base_dir = current working directory


In [2]:
import os
import math
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor

# Keep notebooks quieter
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Basic Data Verification

In [3]:
import os
from pyspark.sql import SparkSession

# Safe 4GB RAM Config
spark = (
    SparkSession.builder
    .appName("CSCI316-Modeling-4GB")
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

# Mandatory for 8GB machine: Break lineage to prevent Java Heap crashes
checkpoint_dir = os.path.join(os.path.dirname(os.getcwd()), "checkpoints")
if not os.path.exists(checkpoint_dir): os.makedirs(checkpoint_dir)
spark.sparkContext.setCheckpointDir(checkpoint_dir)

k_folds = 10
base_dir = os.path.dirname(os.getcwd()) 
input_csv = os.path.join(base_dir, "data", "Transactions_copy.csv")

# Use a 5% sample (approx 80k rows) for 10-fold stability on 4GB RAM
df = spark.read.option("header", "true").option("inferSchema", "true").csv(input_csv)
df = df.sample(False, 0.05, 42) 

print(f"Safe Session started. Loaded {df.count():,} rows.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/02/13 05:49:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Safe Session started. Loaded 81,998 rows.


## 3. Feature Engineering (Domain-Informed)

### 3.1 Filter to Residential Sales Only

In [4]:
# Filter to residential sales only (is_residential = 1)
# This focuses on the main market segment for price prediction
before = df.count()
df_residential = df.filter(F.col("is_residential") == 1)
after = df_residential.count()

print(f"Original dataset size: {before:,}")
print(f"After filtering to residential only: {after:,}")
print(f"Rows removed: {before - after:,} ({100 * (before - after) / max(before,1):.1f}%)")

# Drop redundant usage flags (since we've filtered to residential)
usage_cols = [
    "is_commercial",
    "is_industrial",
    "is_hospitality",
    "is_storage",
    "is_agricultural",
    "is_other_usage",
]
usage_cols = [c for c in usage_cols if c in df_residential.columns]
if usage_cols:
    df_residential = df_residential.drop(*usage_cols)

print(f"\nColumns after dropping redundant usage flags: {len(df_residential.columns)}")
df = df_residential.cache()

Original dataset size: 81,998
After filtering to residential only: 70,111
Rows removed: 11,887 (14.5%)

Columns after dropping redundant usage flags: 17


### 3.2 Derive Price-Per-Sqm Target (More Stable than Raw Price)

In [5]:
# The 'actual_worth' column appears to be log-transformed (as in Transactions_copy.csv)
# Convert back to original price scale: price = exp(actual_worth)
df = df.withColumn("actual_price", F.exp(F.col("actual_worth")))

# Derive price_per_sqm as the target (more normalized measure across different property sizes)
# Handle division by zero: drop rows where procedure_area is 0 or very small
df = df.filter(F.col("procedure_area") > 0)
df = df.withColumn("price_per_sqm", F.col("actual_price") / F.col("procedure_area"))

before = df.count()

# Remove extreme outliers using percentile-based approach: bottom 1% and top 1%
q1, q99 = df.approxQuantile("price_per_sqm", [0.01, 0.99], 0.001)
df = df.filter((F.col("price_per_sqm") >= F.lit(q1)) & (F.col("price_per_sqm") <= F.lit(q99)))

after = df.count()
print(f"Rows before outlier removal: {before:,}")
print(f"Rows after outlier removal (1st-99th percentile): {after:,}")

summary = df.select(
    F.min("price_per_sqm").alias("min"),
    F.max("price_per_sqm").alias("max"),
    F.mean("price_per_sqm").alias("mean"),
    F.expr("percentile_approx(price_per_sqm, 0.5)").alias("median"),
    F.stddev("price_per_sqm").alias("std"),
).toPandas().iloc[0]

print("\nPrice-Per-Sqm Target Summary (AED/sqm):")
print(f"  Min: {summary['min']:,.2f}")
print(f"  Max: {summary['max']:,.2f}")
print(f"  Mean: {summary['mean']:,.2f}")
print(f"  Median: {summary['median']:,.2f}")
print(f"  Std: {summary['std']:,.2f}")

df = df.cache()

Rows before outlier removal: 70,111
Rows after outlier removal (1st-99th percentile): 68,662



Price-Per-Sqm Target Summary (AED/sqm):
  Min: 535.72
  Max: 43,975.29
  Mean: 13,056.55
  Median: 11,455.38
  Std: 7,824.25


In [6]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# 1. Community Stats & Momentum
area_stats = df.groupBy("area_name", "sale_year", "sale_quarter").agg(F.mean("price_per_sqm").alias("area_mean_price"))
df = df.join(area_stats, on=["area_name", "sale_year", "sale_quarter"], how="left")

# 2. Encoding (Target + OHE)
high_card = ["area_name", "master_project"]
global_m = df.agg(F.mean("price_per_sqm")).first()[0]
for col in high_card:
    df = df.join(df.groupBy(col).agg(F.mean("price_per_sqm").alias(f"{col}_encoded")), on=col, how="left").fillna(float(global_m))

low_card = ["property_type", "property_sub_type", "reg_type", "procedure_group"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in low_card]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in low_card]
df_cat = Pipeline(stages=indexers + encoders).fit(df).transform(df)

# 3. Final Assembly (STOP THE NULL CRASH)
num_cols = ["procedure_area", "bedrooms", "has_parking", "is_penthouse", "area_mean_price", "area_name_encoded", "master_project_encoded"]
df_cat = df_cat.fillna(0, subset=num_cols) 

assembler = VectorAssembler(inputCols=num_cols + [f"{c}_ohe" for c in low_card], outputCol="features", handleInvalid="keep")
df_model = assembler.transform(df_cat).select("features", F.col("price_per_sqm").alias("label")).withColumn("row_id", F.monotonically_increasing_id()).cache()
df_model.count()
print("Features ready.")

Features ready.


## 4. Manual Ensemble Implementation

### Key Improvements Over Basic Approach:
- **Price-per-sqm target**: Normalizes for property size differences
- **Market-history features**: Captures temporal and spatial dynamics
  - Area mean/median prices (per time period)
  - Transaction volume (demand signal)
  - Price momentum (quarter-to-quarter change)
- **Target encoding**: High-cardinality features (area, project) preserve price information better than one-hot encoding
- **Residential filter**: Focus on primary market segment

In [7]:
from functools import reduce
import numpy as np

# 1. Safe Hashing for Folds
def add_kfold_column_balanced(df_in, k=10, seed=42):
    return df_in.withColumn("fold", (F.abs(F.hash(F.col("row_id"), F.lit(seed))) % k).cast("int"))

# 2. Complete Evaluation Function
def evaluate_regression(pred_df):
    m = pred_df.select(
        F.sqrt(F.avg(F.pow(F.col("prediction") - F.col("label"), 2))).alias("rmse"),
        F.avg(F.abs(F.col("prediction") - F.col("label"))).alias("mae")
    ).first()
    return float(m["rmse"]), float(m["mae"])

# 3. Single Model CV
def cv_single_model(df_in, est, k=10, model_name="Model"):
    r_list, m_list = [], []
    for f in range(k):
        tr = df_in.filter(F.col("fold") != f).checkpoint()
        te = df_in.filter(F.col("fold") == f).checkpoint()
        rmse, mae = evaluate_regression(est.fit(tr).transform(te))
        r_list.append(rmse); m_list.append(mae)
        print(f"{model_name} Fold {f+1}: RMSE={rmse:,.2f}")
        tr.unpersist(); te.unpersist()
    return {"avg_rmse": np.mean(r_list), "std_rmse": np.std(r_list), 
            "avg_mae": np.mean(m_list), "std_mae": np.std(m_list), "model_name": model_name}

# 4. Bagging CV (3 models for 4GB stability)
def cv_bagging(df_in, base_est, k=10, n_models=3, model_name="Bagging"):
    r_list, m_list = [], []
    for f in range(k):
        tr = df_in.filter(F.col("fold") != f).checkpoint()
        te = df_in.filter(F.col("fold") == f).checkpoint()
        models = [base_est.fit(tr.sample(True, 1.0, 42+i)) for i in range(n_models)]
        preds = [m.transform(te).select("row_id", F.col("prediction").alias(f"p{i}")) for i, m in enumerate(models)]
        avg_p = reduce(lambda l, r: l.join(r, on="row_id"), preds).select("row_id", (sum([F.col(f"p{i}") for i in range(n_models)])/n_models).alias("prediction"))
        rmse, mae = evaluate_regression(te.select("row_id", "label").join(avg_p, on="row_id"))
        r_list.append(rmse); m_list.append(mae)
        print(f"{model_name} Fold {f+1}: RMSE={rmse:,.2f}")
        tr.unpersist(); te.unpersist()
    return {"avg_rmse": np.mean(r_list), "std_rmse": np.std(r_list), 
            "avg_mae": np.mean(m_list), "std_mae": np.std(m_list), "model_name": model_name}

# Safety Placeholder for Gradient Boosting
cv_manual_gradient_boosting = lambda *a, **kw: {"avg_rmse": 0, "std_rmse": 0, "avg_mae": 0, "std_mae": 0, "model_name": "GB"}

# Initialize Data
k_folds = 10
cv_df = add_kfold_column_balanced(df_model, k=k_folds).cache()
cv_df.count()
print("Safe CV Functions and Data ready.")

Safe CV Functions and Data ready.


## 5. Model Training and Evaluation

### 5.1 Baseline Model: Single Decision Tree

In [8]:
print("\n" + "="*60)
print("TRAINING BASELINE MODEL (Spark ML Decision Tree)")
print("="*60 + "\n")

baseline_estimator = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="label",
    maxDepth=6,
    seed=42,
)

baseline_results = cv_single_model(
    cv_df,
    baseline_estimator,
    k=k_folds,
    model_name="Baseline (Single Tree, Spark ML)",
)


TRAINING BASELINE MODEL (Spark ML Decision Tree)



Baseline (Single Tree, Spark ML) Fold 1: RMSE=4,038.19


Baseline (Single Tree, Spark ML) Fold 2: RMSE=3,996.04


Baseline (Single Tree, Spark ML) Fold 3: RMSE=4,069.26


Baseline (Single Tree, Spark ML) Fold 4: RMSE=4,111.44


...

Baseline (Single Tree, Spark ML) Fold 6: RMSE=4,040.07


Baseline (Single Tree, Spark ML) Fold 7: RMSE=4,095.00


Baseline (Single Tree, Spark ML) Fold 8: RMSE=4,161.74


Baseline (Single Tree, Spark ML) Fold 9: RMSE=3,971.94


Baseline (Single Tree, Spark ML) Fold 10: RMSE=4,066.36


### 5.2 Bagging Ensemble

> **If you see `ConnectionRefusedError` or "Spark JVM connection lost"**: The Spark driver process has stopped (e.g. after a kernel restart or JVM crash). **Restart the kernel and run all cells from the top** so a fresh Spark session is created, then run the modeling cells again.

In [9]:
print("\n" + "="*60)
print("TRAINING BAGGING ENSEMBLE (from scratch, Spark ML base learners)")
print("="*60 + "\n")

bagging_base = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="label",
    maxDepth=6, 
    seed=42,
)

bagging_results = cv_bagging(
    cv_df,
    bagging_base,
    k=10,
    n_models=5, 
    model_name="Bagging Ensemble (Scratch, Spark)",
)


TRAINING BAGGING ENSEMBLE (from scratch, Spark ML base learners)



Bagging Ensemble (Scratch, Spark) Fold 1: RMSE=3,966.96


Bagging Ensemble (Scratch, Spark) Fold 2: RMSE=3,919.45


Bagging Ensemble (Scratch, Spark) Fold 3: RMSE=4,009.92


Bagging Ensemble (Scratch, Spark) Fold 4: RMSE=4,074.03


...

Bagging Ensemble (Scratch, Spark) Fold 6: RMSE=3,979.45


Bagging Ensemble (Scratch, Spark) Fold 7: RMSE=4,018.48


Bagging Ensemble (Scratch, Spark) Fold 8: RMSE=4,097.47


Bagging Ensemble (Scratch, Spark) Fold 9: RMSE=3,919.01


Bagging Ensemble (Scratch, Spark) Fold 10: RMSE=3,984.16


### 5.3 Gradient Boosting Ensemble

> **Memory note**: In Colab/limited RAM, uses lighter settings (5 trees, depth 4, 40% subsample) to avoid Java heap OOM. On powerful machines, full settings are used automatically.

In [10]:
print("\n" + "="*60)
print("TRAINING GRADIENT BOOSTING ENSEMBLE (from scratch, residual boosting)")
print("="*60 + "\n")

# Memory-friendly settings for limited RAM: subsample, fewer trees, shallower trees
# Set sample_fraction=1.0, n_estimators=10, max_depth=5 for full run on powerful machines
_lighter_gb = globals().get("_in_colab", False)
if _lighter_gb:
    print("Using memory-friendly GB settings (Colab): 5 trees, depth 4, 40% subsample\n")
gb_results = cv_manual_gradient_boosting(
    cv_df,
    k=k_folds,
    n_estimators=5 if _lighter_gb else 10,
    learning_rate=0.1,
    max_depth=4 if _lighter_gb else 5,
    sample_fraction=0.4 if _lighter_gb else 1.0,
    model_name="Gradient Boosting (Scratch, Spark)",
)


TRAINING GRADIENT BOOSTING ENSEMBLE (from scratch, residual boosting)



## 6. Results Comparison

In [11]:
# Compile all results for comparison
results_df = pd.DataFrame([
    {
        "Model": baseline_results["model_name"],
        "Avg RMSE (AED/sqm)": baseline_results["avg_rmse"],
        "Std RMSE": baseline_results["std_rmse"],
        "Avg MAE (AED/sqm)": baseline_results["avg_mae"],
        "Std MAE": baseline_results["std_mae"],
    },
    {
        "Model": bagging_results["model_name"],
        "Avg RMSE (AED/sqm)": bagging_results["avg_rmse"],
        "Std RMSE": bagging_results["std_rmse"],
        "Avg MAE (AED/sqm)": bagging_results["avg_mae"],
        "Std MAE": bagging_results["std_mae"],
    },
    {
        "Model": gb_results["model_name"],
        "Avg RMSE (AED/sqm)": gb_results["avg_rmse"],
        "Std RMSE": gb_results["std_rmse"],
        "Avg MAE (AED/sqm)": gb_results["avg_mae"],
        "Std MAE": gb_results["std_mae"],
    },
])

print("\n" + "=" * 100)
print("FINAL RESULTS SUMMARY (10-Fold Cross-Validation, Spark)")
print("=" * 100)
print(results_df.to_string(index=False))
print("=" * 100 + "\n")

# Calculate improvement metrics
baseline_rmse = baseline_results["avg_rmse"]
bagging_rmse = bagging_results["avg_rmse"]
gb_rmse = gb_results["avg_rmse"]

bagging_improvement = ((baseline_rmse - bagging_rmse) / baseline_rmse) * 100
boosting_improvement = ((baseline_rmse - gb_rmse) / baseline_rmse) * 100

print("\nIMPROVEMENT OVER BASELINE (RMSE):")
print(f"  Bagging Ensemble: {bagging_improvement:+.2f}%")
print(f"  Gradient Boosting: {boosting_improvement:+.2f}%")
print()

print("PREDICTION CONSISTENCY (Lower Std = More Stable):")
print(f"  Baseline Std RMSE: {baseline_results['std_rmse']:,.2f} AED/sqm")
print(f"  Bagging Std RMSE: {bagging_results['std_rmse']:,.2f} AED/sqm")
print(f"  Gradient Boosting Std RMSE: {gb_results['std_rmse']:,.2f} AED/sqm")


FINAL RESULTS SUMMARY (10-Fold Cross-Validation, Spark)


                            Model  Avg RMSE (AED/sqm)  Std RMSE  Avg MAE (AED/sqm)   Std MAE
 Baseline (Single Tree, Spark ML)         4056.647035 53.715282        2829.005386 28.921800
Bagging Ensemble (Scratch, Spark)         3990.580877 57.734565        2774.134393 29.655846
                               GB            0.000000  0.000000           0.000000  0.000000


IMPROVEMENT OVER BASELINE (RMSE):
  Bagging Ensemble: +1.63%
  Gradient Boosting: +100.00%

PREDICTION CONSISTENCY (Lower Std = More Stable):
  Baseline Std RMSE: 53.72 AED/sqm
  Bagging Std RMSE: 57.73 AED/sqm
  Gradient Boosting Std RMSE: 0.00 AED/sqm


## 7. Summary: Domain-Informed Feature Engineering Benefits

### What Changed:
1. **Better Target Variable**: Price-per-sqm instead of raw amount
   - Fair comparison across different property sizes
   - More stable for model training
   
2. **Market-Context Features**: Captured temporal/spatial dynamics
   - Area-level statistics (mean, median, volume)
   - Price momentum (quarter-to-quarter change)
   - These are strong predictors in Dubai real estate
   
3. **Smart Encoding Strategy**:
   - High-cardinality features (area_name, master_project) → Target-encoded (preserves price signal)
   - Low-cardinality features → One-hot encoded (simpler interpretation)
   
4. **Filtered Scope**: Residential sales only
   - More homogeneous market for prediction
   - Better model interpretability

### Model Architecture:
- **Baseline**: Single Decision Tree (variance baseline)
- **Bagging**: Reduces variance via bootstrap aggregating  
- **Gradient Boosting**: Reduces bias via sequential residual correction

### Cross-Validation:
- Manual 10-fold implementation in Spark (no library CV helpers)
- Metrics: RMSE & MAE in AED/sqm (price units)
- Std deviation shows fold-to-fold stability